# Notebook 04 – Subgraph Mini-Batching Setup

**Task 1.1 – Data Loader & Link Predictor Engineering**

This notebook implements `LinkNeighborLoader` for scalable mini-batch link prediction on the heterogeneous graph built in Notebook 03.

## What this notebook does
1. Loads the pre-saved `HeteroData` object (`hetero_data.pt`).
2. Adds `edge_label_index` / `edge_label` attributes to the `('user', 'reviews', 'item')` relation for the **training** split.
3. Instantiates `LinkNeighborLoader` with:
   - `num_neighbors=[15, 10]` → 2-hop neighbourhood sampling.
   - `neg_sampling_ratio=1.0` → dynamic negative sampling (1 negative per positive edge per batch).
4. Runs a validation loop to inspect the first few mini-batches.
5. Saves the loader configuration as a reference checkpoint.

In [1]:
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import LinkNeighborLoader

PROCESSED_DIR = '../data/processed/'

print(f"PyTorch version    : {torch.__version__}")

import torch_geometric
print(f"PyG version        : {torch_geometric.__version__}")

PyTorch version    : 2.11.0+cpu
PyG version        : 2.7.0


## 1. Load the HeteroData Object

In [2]:
# Load the HeteroData object produced by 03_hetero_data.ipynb
data = torch.load(PROCESSED_DIR + 'hetero_data.pt', weights_only=False)
print(data)

print("\n=== Node & Edge Counts ===")
print(f"  Users : {data['user'].num_nodes}")
print(f"  Items : {data['item'].num_nodes}")
print(f"  User→Item review edges : {data['user', 'reviews', 'item'].edge_index.shape[1]}")
print(f"  Item→Item also_bought edges : {data['item', 'also_bought', 'item'].edge_index.shape[1]}")

HeteroData(
  user={ x=[1642, 384] },
  item={ x=[21639, 384] },
  (user, reviews, item)={ edge_index=[2, 10751] },
  (item, also_bought, item)={ edge_index=[2, 7253] }
)

=== Node & Edge Counts ===
  Users : 1642
  Items : 21639
  User→Item review edges : 10751
  Item→Item also_bought edges : 7253


## 2. Define `edge_label_index` for the Training Split

For link prediction we need to split the `('user', 'reviews', 'item')` edges into
**training** (message-passing) and **supervision** (label) sets.

Here we use an 80/20 random split:
- 80 % → message-passing edges (`edge_index`) that the GNN may use to aggregate.
- 20 % → supervision edges (`edge_label_index`) that the loader will sample from batch-by-batch.

All positive supervision edges receive label `1`. Negative edges are generated
**dynamically** inside `LinkNeighborLoader` via `neg_sampling_ratio`.

In [3]:
import torch
from torch_geometric.transforms import RandomLinkSplit

# ---------------------------------------------------------------------------
# RandomLinkSplit automatically:
#   • Splits the target edge type into train / val / test.
#   • Populates edge_label_index and edge_label on each split.
#   • Ensures the message-passing graph does NOT contain supervision edges
#     (disjoint split mode).
# ---------------------------------------------------------------------------
transform = RandomLinkSplit(
    num_val=0.1,                                   # 10 % → validation
    num_test=0.1,                                  # 10 % → test
    disjoint_train_ratio=0.2,                      # 20 % of train edges → supervision
    neg_sampling_ratio=0.0,                        # Negatives handled by LinkNeighborLoader
    add_negative_train_samples=False,
    edge_types=('user', 'reviews', 'item'),
    rev_edge_types=None,                           # No reverse edges needed for bipartite
)

train_data, val_data, test_data = transform(data)

print("=== After RandomLinkSplit ===")
print(f"train_data edge_label_index shape : "
      f"{train_data['user', 'reviews', 'item'].edge_label_index.shape}")
print(f"train_data edge_label shape       : "
      f"{train_data['user', 'reviews', 'item'].edge_label.shape}")
print(f"val_data   edge_label_index shape : "
      f"{val_data['user', 'reviews', 'item'].edge_label_index.shape}")
print(f"test_data  edge_label_index shape : "
      f"{test_data['user', 'reviews', 'item'].edge_label_index.shape}")

=== After RandomLinkSplit ===
train_data edge_label_index shape : torch.Size([2, 1720])
train_data edge_label shape       : torch.Size([1720])
val_data   edge_label_index shape : torch.Size([2, 1075])
test_data  edge_label_index shape : torch.Size([2, 1075])


In [7]:
# Lấy tổng số cạnh ban đầu (trước khi split) để làm mốc 100%
total_original_edges = data['user', 'reviews', 'item'].edge_index.shape[1]

# Lấy số lượng cạnh trong tập train_data
train_msg_edges = train_data['user', 'reviews', 'item'].edge_index.shape[1]
train_sup_edges = train_data['user', 'reviews', 'item'].edge_label_index.shape[1]

print("=== KIỂM CHỨNG TỈ LỆ CHIA DATA ===")
print(f"Tổng số review gốc: {total_original_edges} cạnh (100%)")
print("-" * 40)
print(f"Train - edge_index (Tài liệu học): {train_msg_edges} cạnh")
print(f"-> Chiếm khoảng {train_msg_edges / total_original_edges * 100:.1f}% đồ thị gốc")
print("-" * 40)
print(f"Train - edge_label_index (Bài kiểm tra trên lớp): {train_sup_edges} cạnh")
print(f"-> Chiếm khoảng {train_sup_edges / total_original_edges * 100:.1f}% đồ thị gốc")

=== KIỂM CHỨNG TỈ LỆ CHIA DATA ===
Tổng số review gốc: 10751 cạnh (100%)
----------------------------------------
Train - edge_index (Tài liệu học): 6881 cạnh
-> Chiếm khoảng 64.0% đồ thị gốc
----------------------------------------
Train - edge_label_index (Bài kiểm tra trên lớp): 1720 cạnh
-> Chiếm khoảng 16.0% đồ thị gốc


## 3. Instantiate `LinkNeighborLoader`

**Key parameters**

| Parameter | Value | Explanation |
|---|---|---|
| `edge_label_index` | `('user','reviews','item')` | Target edge type for link prediction |
| `edge_label` | from `train_data` | Positive labels (all 1s before neg sampling) |
| `num_neighbors` | `[15, 10]` | Hop-1: up to 15 neighbours; Hop-2: up to 10 neighbours |
| `neg_sampling_ratio` | `1.0` | 1 negative edge generated per positive edge in each batch |
| `batch_size` | `256` | Number of supervision edges per mini-batch |
| `shuffle` | `True` (train) | Shuffle training batches |


In [4]:
BATCH_SIZE = 256
NUM_NEIGHBORS = [15, 10]     # [hop-1 fanout, hop-2 fanout]
NEG_SAMPLING_RATIO = 1.0     # 1 negative per positive edge

# ----- Training Loader -----
train_loader = LinkNeighborLoader(
    data=train_data,
    num_neighbors=NUM_NEIGHBORS,
    edge_label_index=(
        ('user', 'reviews', 'item'),
        train_data['user', 'reviews', 'item'].edge_label_index,
    ),
    edge_label=train_data['user', 'reviews', 'item'].edge_label,
    neg_sampling_ratio=NEG_SAMPLING_RATIO,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,   # set > 0 for multiprocessing on Linux/macOS
)

# ----- Validation Loader (no negative sampling needed – val labels already set) -----
val_loader = LinkNeighborLoader(
    data=val_data,
    num_neighbors=NUM_NEIGHBORS,
    edge_label_index=(
        ('user', 'reviews', 'item'),
        val_data['user', 'reviews', 'item'].edge_label_index,
    ),
    edge_label=val_data['user', 'reviews', 'item'].edge_label,
    neg_sampling_ratio=0.0,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print("Training loader   :", train_loader)
print("Validation loader :", val_loader)

Training loader   : LinkNeighborLoader()
Validation loader : LinkNeighborLoader()


## 4. Inspect Mini-Batches

Iterate over the first few training batches to verify the shapes and negative samples.

In [5]:
print("=" * 60)
print("Inspecting the first 3 training mini-batches")
print("=" * 60)

for i, batch in enumerate(train_loader):
    if i >= 3:
        break

    uri_edge = batch['user', 'reviews', 'item']
    num_pos = (uri_edge.edge_label == 1).sum().item()
    num_neg = (uri_edge.edge_label == 0).sum().item()

    print(f"\n--- Batch {i} ---")
    print(f"  Sampled users           : {batch['user'].num_nodes}")
    print(f"  Sampled items           : {batch['item'].num_nodes}")
    print(f"  edge_label_index shape  : {uri_edge.edge_label_index.shape}")
    print(f"  Positive edges          : {num_pos}")
    print(f"  Negative edges          : {num_neg}")
    print(f"  edge_label unique vals  : {uri_edge.edge_label.unique().tolist()}")
    print(f"  User feature shape      : {batch['user'].x.shape}")
    print(f"  Item feature shape      : {batch['item'].x.shape}")

print("\nAll checks passed – loader is working correctly.")

Inspecting the first 3 training mini-batches

--- Batch 0 ---
  Sampled users           : 1314
  Sampled items           : 853
  edge_label_index shape  : torch.Size([2, 512])
  Positive edges          : 256
  Negative edges          : 256
  edge_label unique vals  : [0.0, 1.0]
  User feature shape      : torch.Size([1314, 384])
  Item feature shape      : torch.Size([853, 384])

--- Batch 1 ---
  Sampled users           : 1294
  Sampled items           : 903
  edge_label_index shape  : torch.Size([2, 512])
  Positive edges          : 256
  Negative edges          : 256
  edge_label unique vals  : [0.0, 1.0]
  User feature shape      : torch.Size([1294, 384])
  Item feature shape      : torch.Size([903, 384])

--- Batch 2 ---
  Sampled users           : 1312
  Sampled items           : 942
  edge_label_index shape  : torch.Size([2, 512])
  Positive edges          : 256
  Negative edges          : 256
  edge_label unique vals  : [0.0, 1.0]
  User feature shape      : torch.Size([1312, 3

## 5. Summary

| Component | Configuration |
|---|---|
| **Loader class** | `torch_geometric.loader.LinkNeighborLoader` |
| **Target edge type** | `('user', 'reviews', 'item')` |
| **Neighbourhood sampling** | 2-hop: `[15, 10]` |
| **Negative sampling** | Dynamic, ratio `1.0` (1 neg per pos) |
| **Batch size** | 256 supervision edges |
| **Edge split** | 80 % message-passing / 20 % supervision |
| **Val / Test split** | 10 % each, no dynamic neg sampling |

The loaders (`train_loader`, `val_loader`) built here are consumed by the
**GraphSAGE** (Notebook 05) and **GCN** (Notebook 06) training loops.

In [6]:
# Save the split data objects for re-use in downstream notebooks
torch.save(train_data, PROCESSED_DIR + 'train_data.pt')
torch.save(val_data,   PROCESSED_DIR + 'val_data.pt')
torch.save(test_data,  PROCESSED_DIR + 'test_data.pt')

print("Saved:")
print(f"  {PROCESSED_DIR}train_data.pt")
print(f"  {PROCESSED_DIR}val_data.pt")
print(f"  {PROCESSED_DIR}test_data.pt")

Saved:
  ../data/processed/train_data.pt
  ../data/processed/val_data.pt
  ../data/processed/test_data.pt
